<a href="https://colab.research.google.com/github/mr-marius/Machine-Learning---DIO-/blob/main/03_confusion_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Certifique-se de que todas as bibliotecas necessárias estão instaladas
!pip install tensorflow seaborn matplotlib pandas -q

# Importação das bibliotecas
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import seaborn as sns
import pandas as pd


In [2]:
# Diretório de logs (usado pelo TensorBoard no Colab)
diretorio_logs = "/content/logs"

In [3]:
# Carregar o dataset MNIST
(imagens_treino, etiquetas_treino), (imagens_teste, etiquetas_teste) = datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [4]:
# Preparar as imagens para o modelo
imagens_treino = imagens_treino.reshape((60000, 28, 28, 1))
imagens_teste = imagens_teste.reshape((10000, 28, 28, 1))
imagens_treino, imagens_teste = imagens_treino / 255.0, imagens_teste / 255.0

classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [5]:
# Construção do modelo
modelo = models.Sequential()
modelo.add(layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)))
modelo.add(layers.MaxPooling2D((2, 2)))
modelo.add(layers.Conv2D(64, (3, 3), activation="relu"))
modelo.add(layers.MaxPooling2D((2, 2)))
modelo.add(layers.Conv2D(64, (3, 3), activation="relu"))
modelo.add(layers.Flatten())
modelo.add(layers.Dense(64, activation="relu"))
modelo.add(layers.Dense(10, activation="softmax"))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
# Compilação do modelo
modelo.compile(optimizer="adam",
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

In [7]:
# Callback para o TensorBoard
callback_tensorboard = tf.keras.callbacks.TensorBoard(diretorio_logs, histogram_freq=1)

In [ ]:
# Treinar o modelo
modelo.fit(imagens_treino, etiquetas_treino, epochs=5, callbacks=[callback_tensorboard])

Epoch 1/5
 324/1875 ━━━━━━━━━━━━━━━━━━━━ 52s 34ms/step - accuracy: 0.7238 - loss: 0.8734

In [ ]:
# Predições
predicoes = np.argmax(modelo.predict(imagens_teste), axis=-1)
etiquetas_reais = etiquetas_teste

In [ ]:
# Matriz de confusão
matriz_confusao = tf.math.confusion_matrix(labels=etiquetas_reais, predictions=predicoes).numpy()
matriz_confusao_normalizada = np.around(matriz_confusao.astype('float') / matriz_confusao.sum(axis=1)[:, np.newaxis], decimals=2)

matriz_confusao_df = pd.DataFrame(matriz_confusao_normalizada, index=classes, columns=classes)


In [ ]:
# Visualização da matriz de confusão
figura = plt.figure(figsize=(8, 8))
sns.heatmap(matriz_confusao_df, annot=True, cmap=plt.cm.Blues)
plt.tight_layout()
plt.ylabel("Etiqueta real")
plt.xlabel("Etiqueta prevista")
plt.show()

In [ ]:
# Exibir os logs do TensorBoard no Colab
%load_ext tensorboard
%tensorboard --logdir {diretorio_logs}
